In [1]:
import subprocess, sys, os
root = os.path.abspath('')
# macOS: gcc -dynamiclib; Linux: gcc -shared
import platform
flag = '-dynamiclib' if platform.system() == 'Darwin' else '-shared'
ext  = 'dylib'      if platform.system() == 'Darwin' else 'so'
cmd  = ['gcc', '-O2', '-Wall', '-fPIC', flag,
        '-o', f'C/libevoca.{ext}', 'C/evoca.c']
r = subprocess.run(cmd, cwd=root, capture_output=True, text=True)
print(r.stdout or '(no stdout)')
if r.returncode != 0:
    print('STDERR:', r.stderr, file=sys.stderr)
    raise RuntimeError('Build failed')
print('Build OK')

(no stdout)
Build OK


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(''))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import time
from pprint import pprint

from python.evoca_py import EvoCA, make_gol_lut, LUT_BYTES, lut_bit_index
from python.evoca_py import unpack_lut, available_state_init
from python.display  import run as sdl_run
from python.controls import run_with_controls
from python.evoca_py import import_run
from python.controls import available_probes
from python.evoca_explore import evoca_from_scan, evoca_from_scan_top

In [3]:
N = 512
rng3 = np.random.default_rng(7)
gol_lut=make_gol_lut()
sim = EvoCA()
sim.init(N, food_inc=0.0, m_scale=0.0)
sim.set_lut_all(gol_lut)
sim.set_egenome_all(0)
sim.set_v(rng3.integers(0, 2, (N, N), dtype=np.uint8))
sim.set_f_all(0.0)
sim.set_F_all(0.0)

In [4]:
foo = available_probes()
pprint(foo)

{'activity': 'LUT genome activity (scrolling hash-colored strip)',
 'births': 'Mean +/- std of births array',
 'eg_activity': 'Egenome activity (scrolling hash-colored strip)',
 'eg_food': 'Egene food intake (scrolling hash-colored strip; cumulative food '
            'per egene byte, mouthfuls split across max-match-tied winners)',
 'eg_pop': 'Stacked area: egenome population fractions',
 'egene': 'Egene cognitive stats: 3 sub-strips for mean cognitive specificity '
          '(non-* cell-positions per active egene), mean per-cell cognitive '
          'load, and mean food intake per eater',
 'egenome': 'Egenome stats: mean Negene with +/- std band (top) plus three '
            'sub-strips for distinct egene values, mean max-match, and frac at '
            'Negene_max',
 'entropy': 'Local-pattern Shannon entropy',
 'env_food': 'Mean +/- std of environmental food F(x)',
 'lut_complexity': 'Stacked area: LUT ring-dependency level',
 'n_activity': 'N-activity: Channon shadow LUT-hash s

In [9]:
params = {'N':N,
          'food_inc':0.12,
          'm_scale':0.4,
          'mu_lut':0.001,
          'mu_egene':0.1,
          'tax':0.05,
          'gdiff': 4,
          'restricted_mu':False}
params_state = dict(lut='gol', lut_n_init=1,
                  alive='fraction',
                  alive_fraction=0.5,
                  egenome='uniform',
                    egenome_value=0b000011,
                  f_init=0.1,
                  F_init=0.5)
sim.init(**params)
sim.state(**params_state)

probes = {'ts': True,
          'eg_activity': True,
          'activity': True,
          'egenome': True,
          'egene': True
         }
run_with_controls(sim, probes=probes)

EvoCA: ts probe → /Users/n/Projects/EvoCA/ProbeLogs/2026-05-10_144223_ts_manual.csv
EvoCA: egenome probe → /Users/n/Projects/EvoCA/ProbeLogs/2026-05-10_144223_egenome_manual.csv
EvoCA: egene probe → /Users/n/Projects/EvoCA/ProbeLogs/2026-05-10_144223_egene_manual.csv


<Thread(evoca-sim, started daemon 6437744640)>

EvoCA SDL: starting  N=512 px=2  probes=['--eg-activity=psm_52a753b4']
EvoCA SDL: probe SharedMemory open failed: [Errno 2] No such file or directory: '/--activity=psm_4f2eeff2'
EvoCA SDL: activity shm opened (256x512)
EvoCA SDL: eg_activity shm opened (256x512)
EvoCA SDL: egenome shm opened (5x512)
EvoCA SDL: egene shm opened (3x512)
EvoCA SDL: ts shm opened (6 traces)


In [5]:
# Suggested params for seeing egene/egenome evolution.
# Slow mutation lets selection stabilise cognition; mu_egenome>0
# turns on Negene growth; small tax_per_egene gives gentle
# per-position pressure (option-b tax). Watch the egenome
# probe's `Ngene` and the egene probe's `spec`/`load`/`food`
# strips drift over a few thousand ticks.
params = dict(
    N=N,
    food_inc=0.013,
    m_scale=1.2,
    gdiff=0.06,
    mu_lut=0.001,
    mu_egene=0.003,        # 30x slower than the previous 0.1
    mu_egenome=0.005,      # let Negene drift up
    p_dup_egene=1.0,       # default — new slot duplicates a random active one
    tax=0.035,
    tax_per_egene=0.0001,  # gentle per-position pressure (option b)
    tax_lut=0.0,
    restricted_mu=True,
)
params_state = dict(
    lut='gol', lut_n_init=1,
    alive='halfplane',
    egenome='uniform', egenome_value=0b000011,
    f_init=0.5,
    F_init=1.0,
)
sim.init(**params)
sim.state(**params_state)

probes = {
    'ts':       True,
    'eg_activity': True,
    'eg_food':  True,
    'activity': True,
    'egenome':  True,
    'egene':    True,
}
run_with_controls(sim, probes=probes)


EvoCA: ts probe → /Users/n/Projects/EvoCA/ProbeLogs/2026-05-10_154619_ts_manual.csv
EvoCA: egenome probe → /Users/n/Projects/EvoCA/ProbeLogs/2026-05-10_154619_egenome_manual.csv
EvoCA: egene probe → /Users/n/Projects/EvoCA/ProbeLogs/2026-05-10_154619_egene_manual.csv


<Thread(evoca-sim, started daemon 6287503360)>

In [ ]:
# Watch cognition grow from a low starting point.
# Seed with a minimum-survivable specification (centre + axis-1 = 2
# orbits, so spec starts at 5 cell-positions), low tax, and gentle
# per-position pressure. With mu_egenome > 0 the cell can also add
# more egenes; with low mu_egene the existing mask drifts slowly so
# selection has time to act.
params = dict(
    N=N,
    food_inc=0.02,
    m_scale=1.5,
    gdiff=0.05,
    mu_lut=0.0005,
    mu_egene=0.001,
    mu_egenome=0.003,
    p_dup_egene=1.0,
    tax=0.020,
    tax_per_egene=0.00005,   # very gentle: max ≈ 0.01 / tick / cell
    tax_lut=0.0,
    restricted_mu=True,
)
sim.init(**params)
sim.state(lut='gol', lut_n_init=1, alive='halfplane',
          f_init=0.6, F_init=1.0)

# Override the default egenome init (mask=0x3F, fully specified) with
# a small starting cognition: value=0, mask=0b000011 (centre + axis-1
# orbits both expecting 0). spec starts at 1 + 4 = 5.
sim.set_egenome_pair_all(value=0b000000, mask=0b000011)

print('Initial cell (N//2, N//2):', sim.cell_inspect(N//2, N//2))
print('Initial egene_stats:', sim.egene_stats())

probes = {
    'ts':       True,
    'eg_activity': True,
    'eg_food':  True,
    'activity': True,
    'egenome':  True,
    'egene':    True,
}
run_with_controls(sim, probes=probes)


In [9]:
sim.egenome_stats()

{'mean_negene': 2.129810333251953,
 'std_negene': 1.0412410497665405,
 'distinct_egene_values': 63,
 'mean_max_match': 18.293813705444336,
 'frac_at_max': 0.0}

In [11]:
foo = sim.get_egenome()

In [14]:
foo=sim.get_alive()

In [20]:
fooo = foo.flatten()
np.sum(fooo)/len(fooo)

np.float64(0.21223068237304688)

In [21]:
foo=sim.get_egenes_mask()
foo[0][:10]

array([[61, 29, 63, 47, 57,  5, 30, 60],
       [61, 29, 63, 47, 57,  5, 30, 60],
       [62, 63, 60, 63, 13, 61, 47, 61],
       [62, 63, 60, 63, 13, 63, 47, 61],
       [62, 63, 60, 63, 13, 63, 47, 61],
       [62, 63, 60, 63, 13, 63, 47, 61],
       [62, 63, 28, 21,  7,  7, 63,  7],
       [62, 63, 28, 21,  7,  7, 63,  7],
       [62, 63, 28, 21,  7,  7, 63,  7],
       [62, 63, 28, 21,  7,  7, 63,  7]], dtype=uint8)

In [23]:
foo = sim.get_egenome()
foo[0][:10]

array([3, 3, 1, 1, 1, 1, 1, 1, 1, 1], dtype=uint8)

In [ ]:
sim.get_eg